# CNNs With PyTorch Lightning on Landscape Data

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Downloading the Landscape Dataset From a Git Repository

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning torchmetrics timm

In [ ]:
import itertools
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL.Image
import seaborn
import sklearn.metrics
import torch
import torchmetrics
import tqdm.notebook
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

## Preparing the Data

To load our data, we'll combine several libraries: Pillow, NumPy & PyTorch.

In [ ]:
INPUT_SHAPE = (3, 150, 150)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               size: int = INPUT_SHAPE[1],
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> tuple[torch.Tensor, torch.Tensor] | torch.Tensor:
  images = []
  if create_labels:
    labels = []

  # Iterate over the subfolders of the root: each one matches a
  # class
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Processing folders"):

    dir_name = subdir_path.name

    if create_labels:
      # Assign the right label based on the folder name "labels"
      label = label_to_index.get(dir_name)

    # Add each image of the current label (folder) to our dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Folder {dir_name}", leave=False):
      # Use PIL to load the image
      images.append(
          numpy.array(
              PIL.Image.open(image_path).convert("RGB").resize((size, size))))
      if create_labels:
        labels.append(label)

  # PyTorch expects the channels in second position, so we go from the shape
  # (N, H, W, C) to the shape (N, C, H, W)
  images = torch.from_numpy(numpy.array(images)).permute(0, 3, 1, 2).contiguous()
  if create_labels:
    labels = torch.tensor(labels)

  if shuffle:
    perm = torch.randperm(images.shape[0])
    images = images[perm]
    if create_labels:
      labels = labels[perm]

  if create_labels:
    return images, labels
  else:
    return images

In [ ]:
NUM_WORKERS = 2


class LandscapeDataset(Dataset):
  """Image dataset, converted to floats in [0, 1] on the fly.

  With `augment=True` (training only), each read of an image flips it
  horizontally half of the time and crops it at random.
  """

  def __init__(self,
               images: torch.Tensor,
               labels: torch.Tensor | None = None,
               augment: bool = False) -> None:
    self.images = images
    self.labels = labels
    size = images.shape[-1]
    self.augmentation = v2.Compose([
        v2.RandomResizedCrop(size, scale=(0.7, 1.0), antialias=True),
        v2.RandomHorizontalFlip(),
    ]) if augment else None

  def __len__(self) -> int:
    return self.images.shape[0]

  def __getitem__(
      self,
      index: int) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    image = self.images[index]
    if self.augmentation is not None:
      image = self.augmentation(image)
    image = image.float() / 255.
    if self.labels is None:
      return image
    return image, self.labels[index]


def make_loader(images: torch.Tensor,
                labels: torch.Tensor | None = None,
                batch_size: int = 32,
                shuffle: bool = False,
                augment: bool = False) -> DataLoader:
  return DataLoader(LandscapeDataset(images, labels, augment),
                    batch_size=batch_size,
                    shuffle=shuffle,
                    num_workers=NUM_WORKERS,
                    persistent_workers=NUM_WORKERS > 0)


def get_dataloaders(images: torch.Tensor,
                    labels: torch.Tensor,
                    batch_size: int = 32,
                    validation_split: float = 0.3,
                    augment: bool = False
                    ) -> tuple[DataLoader, DataLoader]:
  """Split the data between training and validation."""
  n_validation = int(images.shape[0] * validation_split)
  n_train = images.shape[0] - n_validation
  return (make_loader(images[:n_train],
                      labels[:n_train],
                      batch_size,
                      shuffle=True,
                      augment=augment),
          make_loader(images[n_train:], labels[n_train:], batch_size))


def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Turn a (C, H, W) tensor into a displayable (H, W, C) array."""
  return image.permute(1, 2, 0).numpy()

## Calling `get_images`

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Images shape: {tuple(images.shape)}")
print(f"Labels shape: {tuple(labels.shape)}")

seaborn.countplot(x=labels.numpy())
plt.title("Counts of the different labels")
plt.ylabel("Count")
plt.xlabel("Label")
plt.show()

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[int(labels[img_index])]

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i, j].imshow(to_displayable(image))
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Creating the Model

Here's an example of a "minimalist" CNN

In [ ]:
class ImageClassifier(lightning.LightningModule):
  """Lightning wrapper shared by all our classification models.

  The wrapped model returns logits: the softmax is applied by the loss
  function during training, and explicitly at prediction
  time.
  """

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-4,
               weight_decay: float = 0.,
               num_classes: int = len(label_names),
               input_size: int = INPUT_SHAPE[1],
               ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, INPUT_SHAPE[0], input_size,
                                           input_size)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    images, labels = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, labels)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: torch.Tensor | tuple[torch.Tensor, ...],
                   batch_index: int) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # AdamW applies weight decay directly to the weights (see the slides);
    # frozen parameters (requires_grad=False) get no gradient and are skipped
    # by the optimizer
    return torch.optim.AdamW(self.parameters(),
                             lr=self.hparams.learning_rate,
                             weight_decay=self.hparams.weight_decay)


def make_trainer(max_epochs: int, name: str) -> lightning.Trainer:
  """Create a Trainer that logs its metrics to a CSV file."""
  return lightning.Trainer(max_epochs=max_epochs,
                           accelerator="auto",
                           devices=1,
                           logger=CSVLogger("logs", name=name),
                           enable_checkpointing=False,
                           num_sanity_val_steps=0,
                           log_every_n_steps=10)

In [ ]:
# Initialize and define the model

# The model is a stack of layers where the data flow is sequential
model = nn.Sequential(
    # A first layer of neurons of 1 convolution of 3x3 pixels, taking
    # the 3 channels of the image as input, followed by its activation
    nn.Conv2d(3, 1, kernel_size=3),
    nn.ReLU(),
    # A max pooling layer
    nn.MaxPool2d(3, 3),
    # A tensor manipulation layer: removes all the dimensions except
    # the batch one and another one that contains all the values
    nn.Flatten(),
    # A dense output layer with 6 neurons (the softmax is applied by the
    # loss function)
    nn.Linear(49 * 49, 6),
)

# Wrap the model: the LightningModule defines the loss function and
# the optimizer
classifier = ImageClassifier(model, learning_rate=0.0001)

# Display a summary of the model
print(ModelSummary(classifier, max_depth=-1))

## Can You Explain the Different Numbers of Parameters?

### Solution

First layer of 1 convolution: (kernel size) * (nb kernels) * (nb input channels) + (nb biases (= nb kernels)) = (3 * 3) * 1 * 3 + 1

Last dense layer: (input dim) * (output dim) + (nb biases) = 2401 * 6 + 6

## Training

Let's train this model on our data! At first, we train for a single epoch, just to check that our model works.

In [ ]:
# Train the model
train_loader, validation_loader = get_dataloaders(images, labels)
trainer = make_trainer(max_epochs=1, name="cnn_minimaliste")
trainer.fit(classifier, train_loader, validation_loader)

## Improve This Performance

Take inspiration from the previous model by adding layers, making layers smaller or larger.

Aim for 10 to 20 epochs and less than 1 minute per epoch (for obvious reasons).

You can consider using a dropout layer right before the last dense layer to improve regularization.

You can get an accuracy above 70% on the validation set in a reasonable time.

The proposed solution takes $\approx$ 45 seconds per epoch for 15 epochs and reaches around 85% accuracy on the validation set.

In [ ]:
# Your improvements here
model = nn.Sequential(
    nn.Conv2d(3, 10, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(3, 3),
    nn.Flatten(),
    nn.Linear(10 * 49 * 49, 6),
)
classifier = ImageClassifier(model, learning_rate=0.0001)

# Display a summary of the model
print(ModelSummary(classifier, max_depth=-1))

# Train the model
train_loader, validation_loader = get_dataloaders(images, labels)
trainer = make_trainer(max_epochs=10, name="ameliorations")
trainer.fit(classifier, train_loader, validation_loader)


# Plot the training metrics
def plot_metrics(trainer: lightning.Trainer) -> None:
  # Training and validation metrics aren't written on the same rows
  # of the CSV file: group them by epoch
  metrics = pandas.read_csv(
      pathlib.Path(trainer.logger.log_dir) / "metrics.csv").groupby(
          "epoch").mean(numeric_only=True).dropna(
              subset=["train_loss", "val_loss"])

  plt.plot(metrics.index, metrics["train_accuracy"])
  plt.plot(metrics.index, metrics["val_accuracy"])
  plt.title("Model accuracy")
  plt.ylabel("Accuracy")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper left")
  plt.show()

  plt.plot(metrics.index, metrics["train_loss"])
  plt.plot(metrics.index, metrics["val_loss"])
  plt.title("Model loss")
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper right")
  plt.show()


plot_metrics(trainer)

### Solution

In [ ]:
def conv(in_channels: int, out_channels: int = 200) -> nn.Sequential:
  convolution = nn.Conv2d(in_channels,
                          out_channels,
                          kernel_size=3,
                          padding="same")
  nn.init.orthogonal_(convolution.weight)
  return nn.Sequential(convolution, nn.ReLU())


def pooling() -> nn.MaxPool2d:
  return nn.MaxPool2d(2, 2, ceil_mode=True)


def dropout(rate: float = 0.2) -> nn.Dropout:
  return nn.Dropout(rate)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.ReLU())


def deep_cnn() -> nn.Sequential:
  """The solution's deep CNN, with new weights on every call."""
  return nn.Sequential(
      conv(3),
      pooling(),
      conv(200),
      pooling(),
      conv(200),
      pooling(),
      conv(200),
      pooling(),
      conv(200),
      pooling(),
      conv(200),
      pooling(),
      conv(200),
      nn.Flatten(),
      dropout(),
      dense(200 * 3 * 3, 200),
      dropout(),
      dense(200, 100),
      dropout(),
      dense(100, 50),
      dropout(),
      nn.Linear(50, 6),
  )


model = deep_cnn()

classifier = ImageClassifier(model, learning_rate=0.0001)

# Display a summary of the model
print(ModelSummary(classifier, max_depth=-1))

In [ ]:
# Train the model
train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
trainer = make_trainer(max_epochs=15, name="cnn_profond")
trainer.fit(classifier, train_loader, validation_loader)

# Visualize the training metrics
plot_metrics(trainer)

## Data Augmentation

When the training accuracy keeps rising while the validation accuracy stalls, the model overfits. Data augmentation (see the slides on data loading) shows it slightly modified versions of the training images at every epoch: with `get_dataloaders(..., augment=True)`, each training image is flipped horizontally half of the time and cropped at random. The validation images are left untouched.

Train the same architecture again, with new weights (`deep_cnn()`), on augmented data. Does the gap between the training and validation curves shrink?

In [ ]:
# Your code here

### Solution

In [ ]:
classifier = ImageClassifier(deep_cnn(), learning_rate=0.0001)
train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128,
                                                  augment=True)
trainer = make_trainer(max_epochs=15, name="cnn_profond_augmente")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)

Every epoch sees different versions of the images: the training accuracy rises more slowly and the gap with validation shrinks. The model has to learn features that withstand cropping and orientation, instead of memorizing the training images. The rest of the notebook analyzes this model.

An implementation of [LeNet](https://en.wikipedia.org/wiki/LeNet):

In [ ]:
def conv(in_channels: int, out_channels: int, padding: str) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=padding),
      nn.Sigmoid())


def pooling() -> nn.AvgPool2d:
  return nn.AvgPool2d(2)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.Sigmoid())


le_net = nn.Sequential(
    conv(3, 6, "same"),
    pooling(),
    conv(6, 16, "valid"),
    pooling(),
    nn.Flatten(),
    dense(16 * 35 * 35, 120),
    dense(120, 84),
    nn.Linear(84, 6),
)
le_net_classifier = ImageClassifier(le_net, learning_rate=1e-4)
print(ModelSummary(le_net_classifier, max_depth=-1))

train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
le_net_trainer = make_trainer(max_epochs=15, name="le_net")
le_net_trainer.fit(le_net_classifier, train_loader, validation_loader)
plot_metrics(le_net_trainer)

An implementation of [AlexNet](https://en.wikipedia.org/wiki/AlexNet):

In [ ]:
def conv(in_channels: int,
         out_channels: int,
         kernel_size: int,
         padding: str = "same",
         stride: int = 1) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding), nn.ReLU())


def pooling() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=3, stride=2)


def dropout(rate: float = 0.5) -> nn.Dropout:
  return nn.Dropout(rate)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.ReLU())


alex_net = nn.Sequential(
    conv(3, 96, 11, "valid", 4),
    pooling(),
    conv(96, 256, 5),
    pooling(),
    conv(256, 384, 3),
    conv(384, 384, 3),
    conv(384, 256, 3),
    pooling(),
    nn.Flatten(),
    dense(256 * 3 * 3, 4096),
    dropout(),
    dense(4096, 4096),
    dropout(),
    nn.Linear(4096, 6),
)
alex_net_classifier = ImageClassifier(alex_net, learning_rate=1e-4)
print(ModelSummary(alex_net_classifier, max_depth=-1))

train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
alex_net_trainer = make_trainer(max_epochs=15, name="alex_net")
alex_net_trainer.fit(alex_net_classifier, train_loader, validation_loader)
plot_metrics(alex_net_trainer)

An implementation of an [Inception](https://towardsdatascience.com/a-simple-guide-to-the-versions-of-the-inception-network-7fc52b863202) block (to build a complete network, you'd need to arrange several of them). This implementation defines a custom [PyTorch module](https://docs.pytorch.org/docs/stable/notes/modules.html), since the data flow is no longer sequential:

In [ ]:
class InceptionBlock(nn.Module):
  """Inception block: four branches concatenated along the channel dimension."""

  def __init__(self, in_channels: int, out_channels: int = 64) -> None:
    super().__init__()

    # conv 1x1
    self.conv11 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU())

    # conv 1x1 then conv 3x3
    self.conv33 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding="same"), nn.ReLU())

    # conv 1x1 then conv 5x5
    self.conv55 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 5, padding="same"), nn.ReLU())

    # max pool 3x3 strides 1x1 then conv 1x1
    self.max_pool = nn.Sequential(
        nn.MaxPool2d(3, 1, padding=1),
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU())

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    # concatenation
    return torch.cat([
        self.conv11(inputs),
        self.conv33(inputs),
        self.conv55(inputs),
        self.max_pool(inputs),
    ], dim=1)


def inception() -> ImageClassifier:
  model = nn.Sequential(
      InceptionBlock(INPUT_SHAPE[0]),
      nn.Flatten(),
      nn.Linear(4 * 64 * 150 * 150, 6),
  )
  classifier = ImageClassifier(model, learning_rate=0.0001)
  print(ModelSummary(classifier, max_depth=-1))
  return classifier


inception_classifier = inception()
train_loader, validation_loader = get_dataloaders(images, labels)
inception_trainer = make_trainer(max_epochs=10, name="inception")
inception_trainer.fit(inception_classifier, train_loader, validation_loader)

## Evaluating the Performance on the Test Set

The `seg_test` folder contains a dataset never seen during training.

We'll use the `Trainer`'s `test(model, dataloaders)` method to evaluate the quality of our predictions on this dataset.

In [ ]:
test_images, test_labels = get_images(
    pathlib.Path("landscape") / "seg_test")
test_loader = make_loader(test_images, test_labels, batch_size=128)

# The Trainer that trained the model is also the one that evaluates it
trainer.test(classifier, dataloaders=test_loader)


def predict_probabilities(trainer: lightning.Trainer,
                          classifier: ImageClassifier,
                          images: torch.Tensor,
                          batch_size: int = 128) -> torch.Tensor:
  """Return the probability assigned to each class for each image."""
  return torch.cat(
      trainer.predict(classifier, make_loader(images, batch_size=batch_size)))

## Error Analysis

We display the confusion matrix, then look at misclassified images.

In [ ]:
def analyze_preds(preds: torch.Tensor, labels: torch.Tensor) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  annot=True,
                  fmt="d",
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names)
  plt.title("Confusion matrix")
  plt.show()

  seaborn.countplot(x=list(map(lambda x: label_names[x], preds.tolist())))
  plt.title("Counts of the predicted classes")
  plt.ylabel("Count")
  plt.xlabel("Class")
  plt.show()


test_pred = predict_probabilities(trainer, classifier,
                                  test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  print(f"Prediction: {predicted_class}, true class: {true_class}")
  mistakes = test_images[(test_pred == label_names.index(predicted_class))
                         & (test_labels == label_names.index(true_class))]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for i, j in itertools.product(range(5), repeat=2):
    ax[i, j].axis("off")
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(to_displayable(mistakes[img_index]))
  plt.show()

In [ ]:
# Plot the images predicted as glacier while their label is mountain
plot_mistakes("glacier", "mountain")

In [ ]:
# Plot the images predicted as glacier while their label is sea
plot_mistakes("glacier", "sea")

In [ ]:
# Plot the images predicted as buildings while their label is sea
plot_mistakes("buildings", "sea")

## Transfer Learning

We reuse a network pre-trained on ImageNet, EfficientNetV2-B0 (`timm` library), as a feature extractor: its weights are frozen, and we only train a new classification head for our 6 classes.

The pre-trained weights expect the same preprocessing as during their training (input size, normalization): `timm.data.resolve_model_data_config` gives it. This network was trained on 192x192 images: we reload our images at that size.

In [ ]:
import timm

base_model = timm.create_model("tf_efficientnetv2_b0.in1k",
                               pretrained=True,
                               num_classes=0,
                               global_pool="")
# Input size, interpolation, normalization mean and standard deviation
data_config = timm.data.resolve_model_data_config(base_model)
print(data_config)

transfer_size = data_config["input_size"][-1]
images, labels = get_images(pathlib.Path("landscape") / "seg_train",
                            size=transfer_size)
test_images, test_labels = get_images(pathlib.Path("landscape") / "seg_test",
                                      size=transfer_size)
test_loader = make_loader(test_images, test_labels, batch_size=128)

In the slides, normalization is a transform (`v2.Normalize`) applied by the `Dataset`. Here, it is the model's first layer (`Normalize` module): the CNNs trained from scratch and the pre-trained model thus share the same `DataLoader`s, which produce images in [0, 1].

The pre-trained network is wrapped in `Frozen`, which freezes its weights **and** keeps it in evaluation mode. `requires_grad=False` isn't enough: `trainer.fit` puts the model in `train()` mode, which would update the statistics of its BatchNorm layers and turn its dropout back on.

In [ ]:
class Normalize(nn.Module):
  """Normalize images already scaled to [0, 1], channel by channel."""

  def __init__(self, mean: tuple[float, ...] | float,
               std: tuple[float, ...] | float) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Wrap a frozen module, kept in evaluation mode."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.eval().requires_grad_(False)

  def train(self, mode: bool = True) -> "Frozen":
    return super().train(False)  # BatchNorm and dropout stay frozen

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


# Size of the pre-trained network's feature maps (6x6 for 192x192 images), to
# size the last layer
with torch.no_grad():
  _, _, feature_height, feature_width = base_model(
      torch.zeros(1, 3, transfer_size, transfer_size)).shape

model = nn.Sequential(
    Normalize(data_config["mean"], data_config["std"]),
    Frozen(base_model),
    nn.Dropout2d(0.1),
    nn.Conv2d(base_model.num_features, 1024, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(1024, 256, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(256, 64, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(64, 16, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Flatten(),
    nn.Linear(16 * feature_height * feature_width, 6),
)

# AdamW's weight decay only affects the weights that get a gradient: those of
# the head, the pre-trained network being frozen
classifier = ImageClassifier(model,
                             learning_rate=0.0001,
                             weight_decay=0.01,
                             input_size=transfer_size)

print(ModelSummary(classifier, max_depth=-1))

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=15, name="transfert")
trainer.fit(classifier, train_loader, validation_loader)

plot_metrics(trainer)

In [ ]:
trainer.test(classifier, dataloaders=test_loader)
test_pred = predict_probabilities(trainer, classifier,
                                  test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "sea")

### Questions

1. How many parameters does this model train, and how many does it freeze? (see its summary)
2. Compare its test accuracy with that of the best CNN trained from scratch.
3. Do the remaining errors (confusion matrix, misclassified images) look like those of the CNN?

### Solution

1. Only the head is trained: about 1.6 M parameters, against 5.9 M frozen ones in EfficientNetV2-B0.
2. The pre-trained model clearly beats the CNNs trained from scratch, from the first epochs: the features it learned on over a million ImageNet images transfer well to our 14,000 landscapes.
3. The remaining confusions are mostly between genuinely close classes (glacier and mountain, buildings and street); many of these images are ambiguous, or even mislabeled.

### Fine-Tuning: Unfreezing the Backbone

Once the head is trained, the whole network can be unfrozen and trained for a few more epochs, with a smaller learning rate for the backbone than for the head (see the "Fine-tuning: Unfreezing the Backbone" slide): its weights are already good, we only want to adjust them.

`FineTuner` below gives the backbone (`model[1]`) its own learning rate. Remove `Frozen` from the model (the frozen network is `model[1].module`), unfreeze its weights, then train for 3 epochs with a learning rate of 1e-5 for the backbone and 1e-4 for the head. Does the validation accuracy improve?

In [ ]:
class FineTuner(ImageClassifier):
  """ImageClassifier whose backbone (`model[1]`) has its own learning rate."""

  def __init__(self, model: nn.Module, backbone_learning_rate: float,
               **kwargs) -> None:
    super().__init__(model, **kwargs)
    self.backbone_learning_rate = backbone_learning_rate

  def configure_optimizers(self) -> torch.optim.Optimizer:
    backbone = list(self.model[1].parameters())
    backbone_ids = {id(parameter) for parameter in backbone}
    head = [p for p in self.parameters() if id(p) not in backbone_ids]
    return torch.optim.AdamW([
        {"params": backbone, "lr": self.backbone_learning_rate},
        {"params": head, "lr": self.hparams.learning_rate},
    ], weight_decay=self.hparams.weight_decay)

In [ ]:
# Your code here

### Solution

In [ ]:
# The backbone replaces its Frozen wrapper, and its weights are unfrozen
model[1] = model[1].module.requires_grad_(True)
classifier = FineTuner(model,
                       backbone_learning_rate=1e-5,
                       learning_rate=1e-4,
                       weight_decay=0.01,
                       input_size=transfer_size)
trainer = make_trainer(max_epochs=3, name="fine_tuning")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)
trainer.test(classifier, dataloaders=test_loader)

The accuracy usually gains a few more points: the backbone adapts to our landscapes. Too high a learning rate for the backbone would instead destroy what it learned on ImageNet. The rest of the notebook uses this model.

## Predicting in "Real" Conditions

The `seg_pred` folder contains unannotated images. We therefore can't properly evaluate the performance on this set.

However, we can display photos and the probabilities our model assigns to each class.

In [ ]:
pred_images = get_images(
    pathlib.Path("landscape") / "seg_pred",
    create_labels=False,
    size=transfer_size)
pred_images.shape

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

# Predict the classes of the 25 selected images
probabilities = predict_probabilities(
    trainer, classifier, pred_images[random_indexes.ravel()]).reshape(5, 5, -1)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Get the image and the predicted class
    image = pred_images[img_index]
    predicted_class = label_names[int(probabilities[i, j].argmax())]

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i * 2, j].imshow(to_displayable(image))
    ax[i * 2, j].set_title(f"Example {img_index} ({predicted_class})")
    ax[i * 2, j].axis('off')

    # Display the prediction distribution on the row below
    ax[i * 2 + 1, j].bar(label_names, probabilities[i, j].numpy())

## Accuracy Depending on the Maximum Probability

Let's now see whether the accuracy varies significantly depending on the maximum probability returned by the model.

In [ ]:
test_probabilities = predict_probabilities(trainer, classifier, test_images)

In [ ]:
def threshold_accuracy(probabilities: torch.Tensor,
                       labels: torch.Tensor,
                       threshold: float
                       ) -> tuple[float, float, float, float]:
  predictions = probabilities.argmax(dim=-1)
  mask_above = probabilities.max(dim=-1).values > threshold
  mask_below = ~mask_above

  n_above = int(mask_above.sum())
  n_below = int(mask_below.sum())

  if n_above:
    above = float(
        (predictions[mask_above] == labels[mask_above]).sum()) / n_above
  else:
    above = 1.
  if n_below:
    below = float(
        (predictions[mask_below] == labels[mask_below]).sum()) / n_below
  else:
    below = 1.
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
    test_probabilities, test_labels, 0.99)
print("Accuracy of the model's predictions with a probability above "
      f"the threshold ({ratio_above * 100:.2f}% of the data): "
      f"{accuracy_above:.2f}")
print("Accuracy of the model's predictions with a probability below "
      f"the threshold ({ratio_below * 100:.2f}% of the data): "
      f"{accuracy_below:.2f}")

xs = numpy.arange(101) / 100
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities,
                                                   test_labels,
                                                   x)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Accuracy")
plt.plot(xs, ratios_above, label="Coverage")
plt.xlabel("Prediction threshold")
plt.title("Accuracy and coverage depending on the prediction threshold")
plt.legend()
plt.show()

## Transfer Learning With Transformers

Same approach with a *Vision Transformer* (ViT) pre-trained on ImageNet-21k, loaded with the `transformers` library: the model is frozen, and we only train a classification head, plugged on the embedding of the classification token. Its preprocessing (size, mean and standard deviation) is read from its `AutoImageProcessor`.

Train this model, evaluate it on the test set, then compare the three approaches: CNN trained from scratch, pre-trained EfficientNetV2, pre-trained ViT.

In [ ]:
import transformers

VIT = "google/vit-base-patch16-224-in21k"
# Preprocessing the ViT expects: size, mean and standard deviation
processor = transformers.AutoImageProcessor.from_pretrained(VIT)
vit_size = processor.size["height"]
images, labels = get_images(pathlib.Path("landscape") / "seg_train",
                            size=vit_size)

In [ ]:
class ViTFeatures(nn.Module):
  """Return the classification token embedding computed by the ViT."""

  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.vit(pixel_values=images).last_hidden_state[:, 0]


vit = transformers.AutoModel.from_pretrained(VIT)

transferred_transformer = nn.Sequential(
    Normalize(processor.image_mean, processor.image_std),
    Frozen(ViTFeatures(vit)),
    nn.Linear(vit.config.hidden_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 6),
)
classifier = ImageClassifier(transferred_transformer,
                             learning_rate=0.001,
                             input_size=vit_size)
print(ModelSummary(classifier, max_depth=-1))

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=15, name="transferred_transformer")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)

In [ ]:
test_images, test_labels = get_images(pathlib.Path("landscape") / "seg_test",
                                      size=vit_size)
test_loader = make_loader(test_images, test_labels, batch_size=128)
trainer.test(classifier, dataloaders=test_loader)
test_pred = predict_probabilities(trainer, classifier,
                                  test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)

### Solution

Both pre-trained models clearly beat the CNNs trained from scratch, for a fraction of the training time: that's the whole point of transfer learning when data is scarce (14,000 images here, against over a million for ImageNet). Between EfficientNetV2 and the ViT the gap is small, but the ViT is much bigger (86 M parameters against 5.9 M): every prediction costs much more. Fine-tuning the backbone usually brings the last few points.